# S1_01 : reproduce the Stage 1 eligible samples and reviews

Run all cells in order, or from the project root use
`python S1/helper/run_notebook.py S1/S1_01_data_preparation.ipynb`.
This notebook verifies the archived 2022–2023 TEDS-D public files, runs the four supporting notebooks in separate Python processes, and checks their saved manifests. It writes a timestamped replication receipt. The residential and outpatient eligible samples remain separate; S1_01A is the decision record. Validation draws and descriptive reviews do not establish final multiple-imputation inference or a causal effect.

In [1]:
from pathlib import Path
from datetime import datetime
from zipfile import ZipFile
from tempfile import TemporaryDirectory
import hashlib, importlib.metadata, json, os, platform, subprocess, sys, time
import nbformat
import pandas as pd
from IPython.display import display

ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'S1/helper/run_notebook.py').is_file())
HOME=ROOT/'S1';RUNNER=HOME/'helper/run_notebook.py';OUTPUTS=HOME/'outputs';REPORTS=HOME/'reports'
sys.path.insert(0,str(HOME/'helper'))
from reporting import Review
review=Review(REPORTS,'S1_01_replication_run')
SPEC=json.loads((HOME/'helper/assets/sample_specification.json').read_text())
ARCHIVES={
  2022:('TEDS-D-2022-DS0001-bndl-data-csv_v1.zip','732ffff1653f9cb0532a94a8133946e06781d39c1e8d9e2925a1790ec637c128'),
  2023:('teds-d-2023-ds0001-bndl-data-csv_v1.zip','ca0b025cba04ae59d2772ab513823644ed4f8e6358531f2052b70ad5c65131b9'),
}
def sha256(path):
    with Path(path).open('rb') as stream:return hashlib.file_digest(stream,'sha256').hexdigest()
inputs=[]
for year,(name,archive_hash) in ARCHIVES.items():
    archive=ROOT/'data/raw'/name
    assert archive.is_file(),f'Missing archived public input: {archive}'
    assert sha256(archive)==archive_hash,f'Archived public input changed: {archive}'
    member=f'tedsd_puf_{year}.csv';csv=ROOT/'data/interim/csv'/member
    expected=SPEC['raw_sha256'][str(csv.relative_to(ROOT))]
    with ZipFile(archive) as z:
        assert z.namelist()==[member],f'Unexpected archive member in {archive}'
        with z.open(member) as stream:assert hashlib.file_digest(stream,'sha256').hexdigest()==expected,f'Archive member changed: {member}'
        if not csv.exists():
            csv.parent.mkdir(parents=True,exist_ok=True)
            with TemporaryDirectory(dir=csv.parent,prefix='s1_extract_') as temp:
                extracted=Path(z.extract(member,temp))
                assert sha256(extracted)==expected
                extracted.replace(csv)
    assert sha256(csv)==expected,f'Extracted public input changed: {csv}'
    inputs.append({'year':year,'archive':str(archive.relative_to(ROOT)),'archive_sha256':archive_hash,
                   'csv':str(csv.relative_to(ROOT)),'csv_sha256':expected})
print('Verified original public ZIPs and extracted CSV inputs:')
display(pd.DataFrame(inputs))
review.table('Verified public inputs',pd.DataFrame(inputs));review.mark('Verify or extract archived inputs')

Verified original public ZIPs and extracted CSV inputs:


,year,archive,archive_sha256,csv,csv_sha256
0,2022,data/raw/TEDS-D-2022-DS0001-bndl-data-csv_v1.zip,732ffff1653f9cb0532a94a8133946e06781d39c1e8d9e...,data/interim/csv/tedsd_puf_2022.csv,c254c53fb9ae6fbdce9b2ad3d1f31c246d2386d6093f71...
1,2023,data/raw/teds-d-2023-ds0001-bndl-data-csv_v1.zip,ca0b025cba04ae59d2772ab513823644ed4f8e6358531f...,data/interim/csv/tedsd_puf_2023.csv,8073f862775c60e538ff0a607bd521339d1eca9189e1f4...


## Run the four supporting notebooks in sequence

Each stage runs in a separate Python process. A failure stops the sequence and leaves prior timestamped outputs intact.

In [2]:
SUPPORT=[
 ('residential_preparation','S1_01_01_residential_data_preparation.ipynb','S1_01_data_preparation'),
 ('residential_review','S1_01_02_residential_outcome_review.ipynb','S1_02_outcome_review'),
 ('outpatient_preparation','S1_01_03_outpatient_data_preparation.ipynb','S1_03_outpatient_data_preparation'),
 ('outpatient_review','S1_01_04_outpatient_outcome_review.ipynb','S1_04_outpatient_outcome_review'),
]
stage_runs={}
def run_stage(label,notebook_name,output_prefix):
    before={p.name for p in OUTPUTS.glob(output_prefix+'_*') if p.is_dir()}
    report_before={p.name for p in REPORTS.glob(Path(notebook_name).stem+'_*_full.html')}
    env=os.environ.copy();env['MPLBACKEND']='Agg'
    print('Running',notebook_name,flush=True)
    subprocess.run([sys.executable,str(RUNNER),str(HOME/notebook_name)],cwd=ROOT,env=env,check=True)
    created=[p for p in OUTPUTS.glob(output_prefix+'_*') if p.is_dir() and p.name not in before]
    reports=[p for p in REPORTS.glob(Path(notebook_name).stem+'_*_full.html') if p.name not in report_before]
    assert len(created)==1 and len(reports)==1,f'Expected one new output and HTML review for {notebook_name}'
    stage_runs[label]={'notebook':notebook_name,'output':created[0].name,'html_report':str(reports[0].relative_to(ROOT))}
    review.note(f"**{label}:** {created[0].name}; full execution report `{reports[0].relative_to(ROOT)}`")
    review.mark(label)
    print('Completed',label,'->',created[0],flush=True)

In [3]:
run_stage(*next(stage for stage in SUPPORT if stage[0]=='residential_preparation'))

Running S1_01_01_residential_data_preparation.ipynb
Completed residential_preparation -> /Users/researcher/code_projects/Causal Discovery /code/project_code/notebooks/code_0927/S1/outputs/S1_01_data_preparation_20260926_163212_739830


In [4]:
run_stage(*next(stage for stage in SUPPORT if stage[0]=='residential_review'))

Running S1_01_02_residential_outcome_review.ipynb
Completed residential_review -> /Users/researcher/code_projects/Causal Discovery /code/project_code/notebooks/code_0927/S1/outputs/S1_02_outcome_review_20260926_175955


In [5]:
run_stage(*next(stage for stage in SUPPORT if stage[0]=='outpatient_preparation'))

Running S1_01_03_outpatient_data_preparation.ipynb
Completed outpatient_preparation -> /Users/researcher/code_projects/Causal Discovery /code/project_code/notebooks/code_0927/S1/outputs/S1_03_outpatient_data_preparation_20260926_180001_385313


In [6]:
run_stage(*next(stage for stage in SUPPORT if stage[0]=='outpatient_review'))

Running S1_01_04_outpatient_outcome_review.ipynb
Completed outpatient_review -> /Users/researcher/code_projects/Causal Discovery /code/project_code/notebooks/code_0927/S1/outputs/S1_04_outpatient_outcome_review_20260926_193521_946088


## Verify outputs and write the replication receipt

This checks hashes and links between the new runs. The checks verify the saved files; imputation-based inference is performed separately in S1_03.

In [7]:
def stage_path(label):return OUTPUTS/stage_runs[label]['output']
r1,r2,r3,r4=[stage_path(label) for label,_,_ in SUPPORT]
m1=json.loads((r1/'manifest.json').read_text())
assert m1['status']=='verified_samples_and_imputation_draws'
for name,info in m1['outputs'].items():
    assert sha256(r1/name)==info['sha256'],f'Residential output hash differs: {name}'
for outcome,n in [('completion',241118),('frequency',146194)]:
    assert m1['outputs'][f'{outcome}_final_eligible.csv.gz']['n']==n
m2=json.loads((r2/'manifest.json').read_text())
assert m2['status']=='descriptive_review_complete_inference_pending'
assert m2['source_run']==r1.name and m2['source_manifest_sha256']==sha256(r1/'manifest.json')
for name,digest in m2['outputs'].items():assert sha256(r2/name)==digest
m3=json.loads((r3/'manifest.json').read_text())
assert m3['status'] in {'candidate_membership_and_four_imputation_draws_verified','candidate_membership_and_four_imputation_draws_verified_overlap_review_required'}
assert m3['analysis_role']=='primary_14_state' and len(m3['states'])==14
assert {(s['outcome'],s['n']) for s in m3['samples']}=={('completion',386282),('frequency',152566)}
assert len(m3['imputation']['draws'])==4 and len(m3['propensity_figures'])==2
for item in m3['imputation']['draws']+m3['propensity_figures']:
    assert sha256(r3/item['file'])==item['sha256'],f'Outpatient artifact hash differs: {item["file"]}'
for sample in m3['samples']:
    ids=pd.read_csv(r3/f"{sample['outcome']}_final_eligible.csv.gz",usecols=['episode_source_id']).episode_source_id
    assert len(ids)==sample['n'] and ids.is_unique
    membership=hashlib.sha256(('\n'.join(sorted(ids))+'\n').encode()).hexdigest()
    assert membership==sample['membership_sha256']
m4=json.loads((r4/'manifest.json').read_text())
assert m4['status']=='descriptive_review_complete_readiness_issues_identified'
assert m4['source_run']==r3.name and m4['source_manifest_sha256']==sha256(r3/'manifest.json')
assert sha256(r4/m4['output_file'])==m4['output_sha256']
assert m4['readiness_audit']['frequency_membership_sha256']==next(x['membership_sha256'] for x in m3['samples'] if x['outcome']=='frequency')

samples=pd.DataFrame([
 {'branch':'residential','outcome':outcome,'eligible_episodes':m1['outputs'][f'{outcome}_final_eligible.csv.gz']['n'],'file':str((r1/f'{outcome}_final_eligible.csv.gz').relative_to(ROOT))}
 for outcome in ('completion','frequency')]+[
 {'branch':'outpatient primary (14 states)','outcome':s['outcome'],'eligible_episodes':s['n'],'file':str((r3/f"{s['outcome']}_final_eligible.csv.gz").relative_to(ROOT))}
 for s in m3['samples']])
print('Replicated eligible samples:');display(samples)
print('Pre-imputation outpatient score-tail gate passed:',m3['pre_imputation_diagnostics']['score_tail_gate_passed'])
review.table('Replicated eligible samples',samples)
review.note('**Verification complete.** The samples are integrity-checked eligible episodes. The outpatient frequency score-tail failure, differential frequency observation, outcome-coding differences and inferential uncertainty remain documented; this receipt is not a final causal analysis.')
review.mark('Verify linked manifests and sample membership')
report_path=review.export()
def source_hash(path):
    notebook=nbformat.read(path,as_version=4)
    return hashlib.sha256(json.dumps([c.source for c in notebook.cells],ensure_ascii=False).encode()).hexdigest()
receipt={
 'status':'eligible_samples_and_descriptive_reviews_replicated_inference_pending',
 'created_at':datetime.now().astimezone().isoformat(),
 'raw_inputs':inputs,
 'software':{'python':platform.python_version(),'platform':platform.platform(),
             'packages':{name:importlib.metadata.version(name) for name in ('numpy','pandas','matplotlib','scikit-learn','nbformat','nbconvert')},
             'requirements_lock_sha256':sha256(ROOT/'requirements-stage1.lock.txt')},
 'stages':stage_runs,
 'notebook_source_sha256':{name:source_hash(HOME/name) for _,name,_ in SUPPORT},
 'samples':samples.to_dict('records'),
 'outpatient_pre_imputation_score_tail_gate_passed':m3['pre_imputation_diagnostics']['score_tail_gate_passed'],
 'summary_report':str(report_path.relative_to(ROOT)),
}
receipt_dir=OUTPUTS/f"S1_replication_run_{datetime.now().astimezone():%Y%m%d_%H%M%S_%f}"
receipt_dir.mkdir(parents=True)
(receipt_dir/'manifest.json').write_text(json.dumps(receipt,indent=2)+'\n')
print('Replication receipt:',receipt_dir/'manifest.json')
print('Summary HTML:',report_path)

Replicated eligible samples:
Pre-imputation outpatient score-tail gate passed: False
Replication receipt: /Users/researcher/code_projects/Causal Discovery /code/project_code/notebooks/code_0927/S1/outputs/S1_replication_run_20260926_193528_380348/manifest.json
Summary HTML: /Users/researcher/code_projects/Causal Discovery /code/project_code/notebooks/code_0927/S1/reports/S1_01_replication_run_20260926_193528_353792.html


,branch,outcome,eligible_episodes,file
0,residential,completion,241118,notebooks/code_0927/S1/outputs/S1_01_data_prep...
1,residential,frequency,146194,notebooks/code_0927/S1/outputs/S1_01_data_prep...
2,outpatient primary (14 states),completion,386282,notebooks/code_0927/S1/outputs/S1_03_outpatien...
3,outpatient primary (14 states),frequency,152566,notebooks/code_0927/S1/outputs/S1_03_outpatien...
